# FDM modeling of ARK-cross sections

We use the developed code from "src/ARK_geotop.py"

In [29]:
import os
import sys
from typing import Any
from glob import glob
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt
import pickle

from tools.fdm.src.mfgrid import Grid
from tools.fdm.src.fdm3Blom import Fdm3
from tools.etc.etc import logo

from mf6lab.Projects.ARK_RWS.src.ARK_geotop import Dirs


print(sys.executable)

# --- Needed to make figure separate from the notebook and interactive
%matplotlib qt

# --- Notebook name for logo
NOTEBOOK_NAME = "ARK_fdm.ipynb"
# --- Seet the namespace for the relevant directories
dirs = Dirs()

# --- Get the paths and names of  the geotop pdf files in the order they are in dirs.dino
xsec_paths = {i:name for i, name in enumerate(glob(dirs.dino + '*.pdf'))}
xsec_names = {i:os.path.basename(name) for i, name in enumerate(glob(dirs.dino + '*.pdf'))}

# --- Pickling
def pickleto(var:Any, basename:str, parent:str=dirs.data):
    """Pickle var to os.path.join(dirs.data, basename)"""
    if not basename.endswith('.pkl'):
        basename += ".pkl"

    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'wb') as f:
        print(f"Pickled {basename} --> {parent}")        
        pickle.dump(var, f)

# --- Unpickling
def picklefrom(basename:str, parent:str=dirs.data)->Any:
    """Unpickle varname from os.path.join(parent, basename)"""
    if not basename.endswith(".pkl"):
        basename += ".pkl"
          
    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'rb') as f:
        print(f"Loaded {basename} <-- {parent}")        
        return pickle.load(f)


def spy(idx_arr):
    """Show where the index labels are in the xsec idx_arr."""
    arr = np.squeeze(idx_arr)
        
    fig, ax = plt.subplots(figsize=(10, 6))
    if np.issubdtype(idx_arr.dtype, np.integer):
        title = "Location of legend indices in xsec.idx_arr"
        classes = np.unique(arr)
        cmap = plt.get_cmap('tab20', len(classes) - 1)
    else:
        title = "imshow of given array"
        cmap = plt.get_cmap('viridis')
    ax.set_title(title)
    mappable = ax.imshow(arr, cmap=cmap, origin='upper')
    fig.colorbar(mappable)
    plt.show()

# --- Color for empty legend (empty voxel with geo_unit 'none')
WHITE_01 = np.array([1., 1., 1.])

/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


In [3]:
pprint(xsec_names)

{0: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf',
 1: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '126311,474187.pdf',
 2: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '130049,479466.pdf',
 3: 'BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf',
 4: 'BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf',
 5: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf',
 6: 'BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf',
 7: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '129926,479462.pdf',
 8: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '127484,477893.pdf',
 9: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse '
    '130173,479431.pdf'}


In [4]:
geotop_xsecs = picklefrom("geotop_xsecs.pkl")

Loaded geotop_xsecs.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


## Deal with the second cross section only

In [5]:
isec = 1
xsec = geotop_xsecs[xsec_names[isec]]

print(f"Dealing with xsec {isec}:\n{xsec.name}") 

# --- find the ARK it wronly has index 1 in row 6
ix_ARK = np.where(xsec.idx_arr[6] == 1)[0]

spy(xsec.idx_arr)


Dealing with xsec 1:
BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf


/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_3450/3251956145.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Repair the idx_arr for the ARK canal which looks now filled with material 'a'

The problem is that the idx_arr contains index 1 (anthropogenic) inside the ARK,
which was likely caused by some vertical grid line disturbing the color_match,
so the light gray legend color (1) was matched instead of white (0).

To find ARK look for index = 1 in row 6 to find ARK incision in the xsec.
Then replace all the 1 indices in that column with 0 to make it empty.

In [6]:
idx_arr = xsec.idx_arr.copy()

# --- The columns have idx 1 incorrectly
cols = np.where(idx_arr[1] == 1)[0]

# --- Replace by index 0 ('none')
for j in cols:
    rows = idx_arr[:, j] == 1
    idx_arr[rows, j] = 0
    
# --- Check
spy(idx_arr)

/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_3450/3251956145.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


When this works replace the xsec.idx_arr with the corrected version

In [7]:
# --- Replace origional idx_arr by corrected one
xsec.idx_arr = idx_arr

# --- Check
spy(xsec.idx_arr)
print('idx_arr repaired')

idx_arr repaired


/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_3450/3251956145.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
-xsec.b - xsec.d_damw, -xsec.b, xsec.z_damw, 0

AttributeError: 'Geotop_xsec' object has no attribute 'b'

In [ ]:
gr.xm # [-60< gr.xm < -40]

## Model grid

In [16]:
def set_ARK_xsec(xsec):
    xsec.ground_elev = -1.3
    xsec.stage  = -0.4
    xsec.zbot  = -6.0
    xsec.d_damw = 0.5
    xsec.z_damw = -12.
    xsec.hpp    = xsec.ground_elev - 1
    xsec.c_drainage = 100.
            
    # --- where in the cross section is the ARK
    ixARK = np.where(xsec.idx_arr[10] == 0)[0][0]
    izARK = np.where(xsec.idx_arr[:, ixARK] == 0)[0]
    xsec.zbot = np.round(xsec.zm[izARK[-1]] - xsec.dz / 2, 1)
    
    # --- ARK water extent
    ae = np.round(np.array([xsec.x[ixARK], xsec.x[ixARK+1], xsec.zbot, xsec.stage]), 1)
    
    # --- Half width of ARK:
    xsec.b = (ae[1] - ae[0]) / 2

    # --- Centralize xsec.x around xARKmid
    xsec.xARKmid_orig     = 0.5 * (ae[0] + ae[1])
    xsec.world_extent[:2] -= xsec.xARKmid_orig
    

    # --- Grid x-coordinates (parallel to xsec) with 0 at xsec.xARK_orig
    # --- refined around the edges of the canal xsec.  
    slog = np.logspace(0, 2, 10)

    # --- Horizontal grid refined incide and next to the Canal.
    b = xsec.b

    x_ = np.hstack((
        (b - np.logspace(0, np.log10(b), 10))[::-1], # --- inside ARK, right of middle
        b, b + xsec.d_damw,                                # --- sheet piling
        b + slog[slog > xsec.d_damw],                   # --- increasing cell wirdth first 100 m 
        np.linspace(0, 2000, 21).clip(200, None)        # --- beyond this to 2000 m 100 m cells
    ))

    x_ = np.round(x_, 1)

    # --- Mirror around heart line of ARK and remove doubles   
    x = np.unique(np.hstack((-x_[::-1], x_)))
    
    # --- Get the grid and attach all grid properties to it
    # --- The grid use the new x and the old z
    gr = Grid(x, None, xsec.z[xsec.z <= xsec.ground_elev])
    
    
    # --- ARK-extent --> mask_ARK
    gr.mask_ARK = gr.inblock(xx=(-b, b), yy=None, zz=(xsec.zbot, xsec.stage))
    
    # --- extent of the sheet piling left and right along canal      
    gr.mask_damwL = gr.inblock(xx=(-xsec.b - xsec.d_damw, -xsec.b), zz=(xsec.z_damw, 0))
    gr.mask_damwR = gr.inblock(xx=(+xsec.b, +xsec.b + xsec.d_damw), zz=(xsec.z_damw, 0))

    # --- Drainage        
    gr.mask_DRN = np.logical_and(
        gr.inblock(xx=(gr.x[0], gr.x[-1]), zz=(xsec.hpp - 0.25, xsec.hpp + 0.25)),
        ~gr.mask_ARK
        )
    
    Idrn = gr.NOD[gr.mask_DRN]

    DRN = np.zeros(len(Idrn), dtype=Fdm3.dtypes['drn'])
    DRN['Ig'] = Idrn
    DRN['C'] = xsec.c_drainage
    DRN['h'] = xsec.hpp
    gr.DRN = DRN

    # --- gr Arrays
    IBOUND = gr.const(1, dtype=int)
    IBOUND[gr.mask_damwL] = 0
    IBOUND[gr.mask_damwR] = 0
    IBOUND[gr.mask_ARK]   = -1
    gr.IBOUND = IBOUND
    
    gr.FQ = gr.const(0.)
    
    gr.FH = None
    
    HI = gr.const(xsec.hpp)
    HI[gr.mask_ARK] = xsec.stage
    gr.HI = HI
    
    gr.idx_arr = xsec.overlap(gr)
    
    
    # --- Make the grid propertie arrays 3D
    props  = xsec.get_props(idx_arr=gr.idx_arr)
    for var in props:
        props[var] = props[var][:, np.newaxis, :]
        
    props['kv'][gr.mask_ARK] = 1000.    

    gr.kh = props['kh']
    gr.kv = props['kv']
    gr.K   = (gr.kh, gr.kh, gr.kv)
    gr.n   = props['n']
    gr.rho = props['rho']
    gr.rhow = props['rho_wet']

    gr.S = gr.const(0.001)
    
    xsec.gr = gr
    

set_ARK_xsec(xsec)  

gr = xsec.gr

mdl = Fdm3(gr=xsec.gr, K=gr.K, c=None, S=None, IBOUND=gr.IBOUND, HI=gr.HI, FQ=gr.FQ)
out = mdl.simulate(DRN=gr.DRN, RIV=None, GHB=None, FDR=None, tm=None, htol=1e-7, maxiter=50, verbose=False)
psi = gr.psi_row(out['Qx'])

/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:234: RuntimeWarning: divide by zero encountered in divide
  Rx2 = 0.5 * dx / (dy * dz) / kx
/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:235: RuntimeWarning: divide by zero encountered in divide
  Rx1 = 0.5 * dx / (dy * dz) / kx
/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:236: RuntimeWarning: divide by zero encountered in divide
  Ry  = 0.5 * dy / (dz * dx) / ky
/Users/Theo/Development/python/hydro_tools/tools/fdm/src/fdm3Blom.py:237: RuntimeWarning: divide by zero encountered in divide
  Rz  = 0.5 * dz / (dx * dy) / kz


Non-linear options: ['DRN'], starting outer iterations:
iouter =    0, err =        1.9 m, errBalance =    0.57472
iouter =    1, err =   0.057989 m, errBalance =    0.10477
iouter =    2, err =  0.0091526 m, errBalance =    0.03406
iouter =    3, err = 0.00093167 m, errBalance =   0.020857
iouter =    4, err =  0.0003925 m, errBalance =    0.01514
iouter =    5, err = 0.00016969 m, errBalance =  0.0072431
iouter =    6, err = 7.2543e-05 m, errBalance =  0.0020983
iouter =    7, err = 3.1155e-05 m, errBalance = 0.00030152
iouter =    8, err = 1.3351e-05 m, errBalance = 2.9524e-05
iouter =    9, err = 5.7252e-06 m, errBalance = -1.1345e-06
iouter =   10, err = 2.4539e-06 m, errBalance = 8.8195e-07
iouter =   11, err = 1.0519e-06 m, errBalance = -3.5464e-07
iouter =   12, err =  4.508e-07 m, errBalance = 1.4581e-07
iouter =   13, err = 1.9319e-07 m, errBalance = -6.0027e-08
iouter =   14, err = 8.2785e-08 m, errBalance = 2.4796e-08
Converged, normal termination.

===== Water balance of t

In [30]:
fig, ax = plt.subplots(figsize=(10, 6))

dphi, dpsi = 0.1, 0.1
phi_levels = np.arange(np.floor(xsec.hpp), np.ceil(xsec.stage), dphi)
psi_levels = np.arange(np.floor(psi.min()), np.ceil(psi.max()), dpsi)

fig.suptitle(xsec.name)
ax.set(title=f"Heads and stream lines dPhi={dphi} m dPsi={dpsi} m2/d", xlabel='x van hartlijn ARK', ylabel='z [NAP]')
ax.grid()
ax.contour(gr.xm, gr.zm, out['Phi'][:, 0, :], linewidths=0.5, levels=phi_levels)


ax.contour(gr.x[1:-1], gr.z, psi, colors='r', linewidths=0.5, levels=psi_levels)
ax.set_xlim(-100, 100)
ax.set_aspect(1)

logo(fig, NOTEBOOK_NAME)
fig.savefig(os.path.join(dirs.images, f"ARK_XS{xsec.xRD}{xsec.yRD}.pdf"))

plt.show()

/var/folders/90/m51x_b713y561gzh2kzy18d00000gq/T/ipykernel_3450/3835602069.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
inspect.signature(logo)

<Signature (fig, script_name=None)>

In [ ]:
gr.psi_row(out['Qx'])